# wiki_search: quick start

`WikiQA` answers questions, checks claims and writes short research briefs using only English Wikipedia, with Claude driving the searches and reads. Every fact is cited to the Wikipedia section it came from.

## Setup (once)

```bash
python3 -m venv .venv
.venv/bin/pip install -r requirements.txt
.venv/bin/python -m ipykernel install --user --name wiki-search --display-name "wiki_search (.venv)"
```

Create a `.env` file in the project root:

```
ANTHROPIC_API_KEY=sk-ant-...
WIKI_USER_AGENT=wiki-search/0.1 (your-contact@example.com)
```

Wikimedia asks API clients to identify themselves; with a contact address you get throttled far less. Then open this notebook with the **wiki_search (.venv)** kernel and run the cells in order. Each question takes roughly 10 seconds to 2 minutes, and a few cents of API usage.

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "wiki_search").is_dir())
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

from wiki_search import WikiQA

qa = WikiQA()  # Claude Sonnet 4.6; see the README for options


def show(answer):
    """Display an answer or brief, or the clarifying question if the tool asked one."""
    if answer.needs_clarification:
        c = answer.clarification
        display(Markdown(f"**The tool asks:** {c.question}\n\nOptions: {', '.join(c.options)}"))
    else:
        display(Markdown(answer.text))
    for w in answer.warnings:
        print("⚠️", w)
    print(f"({answer.elapsed_s}s)")

## A fact question

The answer comes first, then the cited evidence and sources.

In [2]:
show(qa.ask("How tall is Aconcagua?"))

**Answer:** **6,967.15 metres (22,858.1 feet)**

**Reasoning:**
- Aconcagua, located in Mendoza Province, Argentina, has a summit elevation of 6,967.15 metres (22,858.1 feet), making it the highest mountain outside Asia. [1]

**Sources:**  
[1] Aconcagua (lead) — https://en.wikipedia.org/wiki/Aconcagua

(9.1s)


## A multi-hop question

When one answer is needed to find the next, the tool resolves the chain hop by hop and cites each step: here, which country the islands belong to, then that country's currency.

In [3]:
show(qa.ask("What currency is used in the country the Galápagos Islands belong to?"))

**Answer:** **The United States dollar (USD)**

**Reasoning:**
1. The Galápagos Islands form the Galápagos Province of the Republic of Ecuador. [1]
2. The present currency of Ecuador is the United States dollar; it also mints its own centavo coins. [2]

**Sources:**  
[1] Galápagos Islands (lead) — https://en.wikipedia.org/wiki/Gal%C3%A1pagos_Islands  
[2] Currency of Ecuador (lead) — https://en.wikipedia.org/wiki/Currency_of_Ecuador

(9.6s)


## A research brief

For overviews, histories and comparisons the tool reads several articles and writes a short brief: an overview, sections, how items were chosen, further reading and sources. Before a brief is accepted, each claim is checked against the text of the section it cites; claims that misstate their source go back to the agent once to fix, and any still flagged after that are printed as warnings below the brief. This takes a minute or two.

In [2]:
show(qa.ask("Give me an overview of how the printing press changed Europe."))

## How the Printing Press Changed Europe

Invented by Johannes Gutenberg around 1440, the printing press unleashed what historians call a 'printing revolution': a cascade of changes in religion, science, politics, literacy, and language that reshaped European civilisation. [1][3] By making text cheap and widely available, the press shifted Europe from an oral, manuscript culture to a reading culture — accelerating movements from the Reformation to the Scientific Revolution. [1][4]

### Invention and Rapid Spread
- Gutenberg developed his movable-type press in Mainz around 1440, combining oil-based ink, adjustable moulds, and a metal alloy of lead, tin, and antimony that melted at a low temperature for fast, economical casting. [2]
- From a single print shop in Mainz, the technology exploded: by the end of the 15th century, around 270 European cities were involved in printing, and by 1500 Western European presses had produced over 20 million copies. [3]
- Output grew to an estimated 150–200 million copies by 1600; European presses of that era could produce between 1,500 and 3,600 impressions per workday. [3]

### Literacy, Authorship, and Reading Habits
- Lower costs and wider availability of printed texts contributed to rising adult literacy rates across Europe over the two centuries following the invention. [4]
- Reading shifted from communal oral recitation to silent, private reading — what historian Elizabeth Eisenstein called a transition 'from a hearing public to a reading public'. [4]
- Authors became more closely identified with their exact words; pages were numbered, indices appeared, and the first copyright laws were passed as book production became commercial. [4]

### The Reformation and Religion
- Martin Luther became Europe's most published author around 1520: 291 editions of his 45 original works were reprinted across Germany, and 300,000 copies distributed between 1518 and 1520. [3]
- Economic research estimates that cities with at least one press by 1500 were 52 percentage points more likely to have adopted Protestantism by 1530 — with the declining effect over subsequent decades suggesting the press mattered most in the Reformation's early phase. [4]
- Authorities tried to contain the press's religious influence — the first Papal Index under Paul IV (1559) forbade lay Catholics to read certain books — but printed material crossed borders easily and clandestine presses flourished. [4]

### Science and Knowledge
- The shift from manuscript to print preceded the great conceptual revolutions of the 16th century: Copernicus's challenge to Ptolemy was itself surpassed within 100 years, a pace historian Elizabeth Eisenstein attributes largely to print. [4]
- Printing enabled scientific communities to form across Europe, sharing methods, serialising new findings, and preserving data across generations — moving away from professional secrecy toward open publication. [4]
- Accurate printed illustrations transformed medicine and botany; hand-copied illustrations had gradually degraded, while printed copies preserved precise visual detail. [4]

### Politics, Power, and Censorship
- Rulers quickly adopted the press: according to historian Elizabeth Eisenstein (citing Geoffrey Elton), Thomas Cromwell's 1530s campaign supporting Henry VIII's break with Rome was the first European government effort to exploit the press for propaganda. [4]
- Print fuelled later political upheavals — pamphlets and broadsheets played a major role in the English Civil War, and later the American and French Revolutions. [4]
- Cities where printing was established in the 15th century grew around 60 percent faster than comparable cities without presses between 1500 and 1600, showing the press had measurable economic effects. [4]

### Language and National Identity
- As books increasingly appeared in regional vernacular languages rather than Latin, printed texts standardised spelling and syntax, reducing regional variability. [1][5]
- Historians Febvre and Martin argue that printing exercised a greater influence on the development of national languages than any other single factor, and identify it as one of several forces behind the rise of nationalism in Europe. [5]

**How these were chosen:** Subtopics were drawn from the 'Impact' section of the Wikipedia 'Printing press' article, which explicitly organises the press's effects into authorship/reading, science, politics/state/church, and language — plus the spread section for historical context. [1][4]

**Further reading:**
- [Protestant Reformation](https://en.wikipedia.org/wiki/Protestant_Reformation) — Explores how Luther's ideas spread and how print shaped the theological and political split in Christianity.
- [Scientific Revolution](https://en.wikipedia.org/wiki/Scientific_Revolution) — Covers how new ideas in astronomy, physics and medicine spread through Europe, with print as a key enabler.
- [Gutenberg Bible](https://en.wikipedia.org/wiki/Gutenberg_Bible) — The first major printed book in Europe — a case study in early print quality and production.
- [History of books](https://en.wikipedia.org/wiki/History_of_books) — Traces how the book as an object evolved from manuscript through print to the digital era.
- [Elizabeth Eisenstein](https://en.wikipedia.org/wiki/Elizabeth_Eisenstein) — Profile of the historian whose landmark work on the press as an 'agent of change' underlies much of this scholarship.
- [Incunabula](https://en.wikipedia.org/wiki/Incunabula) — Books printed before 1501 — the earliest products of European presses and primary sources for this period.

**Sources:**  
[1] Printing press (lead) — https://en.wikipedia.org/wiki/Printing_press  
[2] Johannes Gutenberg (lead) — https://en.wikipedia.org/wiki/Johannes_Gutenberg  
[3] Printing press § Mass production and spread in Europe — https://en.wikipedia.org/wiki/Printing_press#Mass_production_and_spread_in_Europe  
[4] Printing press § Impact — https://en.wikipedia.org/wiki/Printing_press#Impact  
[5] Printing press § Language — https://en.wikipedia.org/wiki/Printing_press#Language

(87.0s)


## Ambiguous names: when the tool asks, and when it doesn't

If a question could reasonably mean several things, the tool asks which one you meant; call `ask` again with the reply. It doesn't ask when the wording or common usage makes one meaning clear.

**Asks:** there are many Springfields, and nothing in the question says which one.

In [5]:
question = "What is the population of Springfield?"
first = qa.ask(question)
show(first)
if first.needs_clarification:
    show(qa.ask(question, clarification='By "Springfield" I mean Springfield, Illinois.'))

**The tool asks:** There are many places named Springfield! Could you clarify which Springfield you're asking about? For example, are you referring to Springfield, Missouri (the most populous city with that name), Springfield, Illinois (the state capital), Springfield, Massachusetts, Springfield, Ohio, or another Springfield entirely?

Options: Springfield, Missouri, Springfield, Illinois, Springfield, Massachusetts, Springfield, Ohio, Springfield, Oregon, Springfield, Virginia

(8.2s)


**Answer:** **114,394 (2020 U.S. Census); ~208,000 in the metro area**

**Reasoning:**
- Springfield, Illinois had a population of 114,394 at the 2020 United States Census, making it the state's seventh-most populous city. [1]
- Approximately 208,000 residents live in the Springfield metropolitan area, which consists of all of Sangamon and Menard counties. [1]

**Sources:**  
[1] Springfield, Illinois (lead) — https://en.wikipedia.org/wiki/Springfield,_Illinois

(8.3s)


**Doesn't ask:** "Macbeth" is also an opera and a Scottish king, but the play is the obvious reading. The tool answers for it and says which meaning it took.

In [6]:
show(qa.ask("When was Macbeth written?"))

**Answer:** **Around 1606 (first performed 1606; first published 1623)**

**Reasoning:**
- Macbeth is estimated to have been first performed in 1606 and was first published in the Folio of 1623. [1]

**Sources:**  
[1] Macbeth (lead) — https://en.wikipedia.org/wiki/Macbeth

(7.8s)
